# 🗄️ 환경 진단 · DB 열기 — Instructor

예측 → 작성 → 실행 → 확인 순서로 강의 코드를 연습한다.

> **🖥️ 환경 설정**
>
> - **로컬**: 수업에서 안내한 FD_26 실행 환경의 커널을 선택한 뒤, 이 노트북을 practice 폴더에서 연다.
>
> - 이번 주는 Colab 병행 없이 VS Code(로컬) 단일 경로다.
>
> **⌨️ 단축키 안내**
>
> | 단축키 | 동작 |
> |--------|------|
> | `Shift + Enter` | 셀 실행 후 다음 셀 이동 |
> | `Esc → A` | 위에 셀 삽입 |
> | `Esc → B` | 아래에 셀 삽입 |
> | `Esc → DD` | 셀 삭제 |


In [1]:
# 🖥️ 환경 설정
# - VS Code에서 FD_26/.venv 커널을 선택했는지 먼저 확인한다 (실행·설치는 전부 FD_26에서).

import os
import sys
import sqlite3
import pandas as pd

print(sys.executable)
# 예상 출력: (자기 프로젝트 경로)/FD_26/.venv/bin/python3

conn = sqlite3.connect('data/market_data.db')


c:\Projects\FD_26\.venv\Scripts\python.exe


## 📁 먼저 파일로 답해본다 — CSV 세 개를 열고 합치는 시도

이 예시의 실행 흐름을 먼저 예측한 뒤, 같은 질문을 노트북에서 확인한다.

[→ §100 강의노트](../notes/FD-03-100__why-database.md)

이 DB 안의 표 세 개를 CSV로 내보내 놓고, 파일만 갖고 100번 앵커 질문("매출이 늘고 있고 산업 평균보다 수익성이 좋은 회사")에 답해본다.


In [3]:
# 📌 §100 📁 CSV 세 개를 열고 합치는 시도 cell_id=100:1
# 이 DB의 표 세 개를 먼저 CSV로 내보낸다 — 파일 기반이었다면 매번 이렇게 받았을 것이다.
for table in ['companies', 'financials', 'returns']:
    pd.read_sql(f"SELECT * FROM {table}", conn).to_csv(f'data/{table}.csv', index=False)

# 이제부터는 DB가 아니라 방금 만든 CSV 세 개만 갖고 있다고 가정한다.
companies_csv = pd.read_csv('data/companies.csv')
financials_csv = pd.read_csv('data/financials.csv')

merged = companies_csv.merge(financials_csv, on='code')

merged[merged['year'] == 2025].sort_values('revenue', ascending=False).head(5)[['name', 'revenue']]

# 예상 출력:
#      name  revenue
#    삼성전자  3466200
#     현대차   1797600
#       기아   1209900
#   SK하이닉스  1007720
#     LG화학    470256
# (companies.csv, financials.csv를 각각 열고 code로 합친 뒤에야 나온 결과다)


,name,revenue
1,삼성전자,3466200
13,현대차,1797600
15,기아,1209900
3,SK하이닉스,1007720
7,LG화학,470256


조건을 하나 바꾸면(예: 연도를 2024로, 업종을 다른 업종으로) 위 셀 전체를 다시 손대야 한다 — 파일을 열고 합치는 절차 자체를 매번 반복한다. 바로 아래 §200부터는 같은 질문을 DB에 SQL로 묻는다.


## 🧪 시작 전에 한 줄

이 예시의 실행 흐름을 먼저 예측한 뒤, 같은 질문을 노트북에서 확인한다.

[→ §200 강의노트](../notes/FD-03-200__sqlite-open.md)


In [ ]:
# 📌 §200-1 🧪 시작 전에 한 줄
# §FD-03-200-1
import sys
print(sys.executable)

# 예상 출력:
# (자기 프로젝트 경로)/FD_26/.venv/bin/python3


## 🔌 DB 파일을 연다

이 예시의 실행 흐름을 먼저 예측한 뒤, 같은 질문을 노트북에서 확인한다.

[→ §200 강의노트](../notes/FD-03-200__sqlite-open.md)


In [4]:
# 📌 §200-2 🔌 DB 파일을 연다
# §FD-03-200-2
import sqlite3

conn=sqlite3.connect('data/market_data.db')
# 예상 출력:
# SQLite 연결 객체가 생성된다.


In [5]:
# 📌 §200-3 🔌 DB 파일을 연다
# §FD-03-200-3
import pandas as pd

pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)

# 예상 출력:
#          name
#     companies
#    financials
#       returns


,name
0,companies
1,financials
2,returns


In [6]:
# 📌 §200-4 🔌 DB 파일을 연다
# §FD-03-200-4
#query=""
#selecet * from companies ""
pd.read_sql("SELECT * FROM companies LIMIT 5", conn)

# 예상 출력:
#      code       name sector market
#    005930     삼성전자   전기전자  KOSPI
#    000660   SK하이닉스   전기전자  KOSPI
#    009150     삼성전기   전기전자  KOSPI
#    051910      LG화학     화학  KOSPI
#    011170    롯데케미칼     화학  KOSPI


,code,name,sector,market
0,005930,삼성전자,전기전자,KOSPI
1,000660,SK하이닉스,전기전자,KOSPI
2,009150,삼성전기,전기전자,KOSPI
3,051910,LG화학,화학,KOSPI
4,011170,롯데케미칼,화학,KOSPI


## 🏗 이 DB는 어떻게 만들어졌나

이 예시의 실행 흐름을 먼저 예측한 뒤, 같은 질문을 노트북에서 확인한다.

`daily_return`은 Week02에서 계산된 CSV 값을 그대로 함께 적재한다. 이번 셀은 수익률을 다시 계산하는 자리가 아니라, 새 거래일을 `append`하고 같은 거래일을 중복 적재하지 않는 흐름을 확인하는 자리다.

[→ §200 강의노트](../notes/FD-03-200__sqlite-open.md)


In [ ]:
# 📌 §200-5 🏗 파이썬 ↔ DB 왕복 티켓: 넣기(`to_sql`)와 꺼내기(`read_sql`)
# §FD-03-200-5
# 새로 들어온 데이터를 내 DB 표에 덧붙여 적재하는 방법
returns_new = pd.read_csv('data/새로_들어온_거래일.csv')
loaded = pd.read_sql("SELECT code FROM returns WHERE date = ?", conn,
                     params=[returns_new["date"].iloc[0]])
if loaded.empty:
    returns_new.to_sql('returns', conn, if_exists='append', index=False)
    print(f"{len(returns_new)}행 적재 (daily_return 포함)")
else:
    print("이 거래일은 이미 적재됨 (daily_return도 중복 적재하지 않음)")
# 예상 출력: 첫 실행은 12행 적재 (daily_return 포함), 다시 실행하면 이미 적재됨 (daily_return도 중복 적재하지 않음).
